<hr style="border:2px solid #0281c9"> </hr>

<img align="left" alt="ESO Logo" src="http://archive.eso.org/i/esologo.png">  

<div align="center">
  <h1 style="color: #0281c9; font-weight: bold;">ESO Science Archive</h1> 
  <h2 style="color: #0281c9; font-weight: bold;">Jupyter Notebooks</h2>
</div>

<hr style="border:2px solid #0281c9"> </hr>

# **Catalogue Data: Cone Search by Position**

This notebook demonstrates how to query ESO catalogue tables with `astroquery.eso` using a positional cone search. 

The example mirrors the style of the other simple notebooks, but uses `query_catalog()` together with an ADQL spatial filter passed through `column_filters`. 

Note that currently the ``ra`` and ``dec`` input parameters to `query_catalog()` are not used for positional filtering, so the ADQL filter is necessary to perform a cone search. 

Here we use the `KiDS_DR4_1_ugriZYJHKs_cat_fits` table and run a small search around `NGC1097`.

<hr style="border:2px solid #0281c9"> </hr>

# **Importing and basic usage of astroquery.eso**

In [1]:
import astroquery # import astroquery
print(f"astroquery version: {astroquery.__version__}") # check the version of astroquery

astroquery version: 0.4.12.dev505+gf2a77a615.d20260427


In [2]:
import astropy.units as u 
from astropy.coordinates import SkyCoord 
from astroquery.eso import Eso 

In [3]:
eso = Eso() # create an instance of the ESO class

# **Defining the catalogue and search position**

Catalogue tables can be very large, so it is useful to keep the returned table small while exploring. In this example we resolve the target name with `SkyCoord.from_name()` and keep the final query to a modest row limit.

In [4]:
catalog_name = "KiDS_DR4_1_ugriZYJHKs_cat_fits" # select one ESO catalogue table
target = "NGC1097" # resolve a target name to coordinates
coords = SkyCoord.from_name(target) # query an online name resolver
radius = 3 * u.arcmin # set the cone-search radius

print(f"Target: {target}")
print(f"Coordinates: {coords.to_string('hmsdms')}")
print(f"Radius: {radius}")

Target: NGC1097
Coordinates: 02h46m19.059s -30d16m29.679996s
Radius: 3.0 arcmin


# **Listing available catalogues**

You can list the catalogue tables known to the archive with `list_catalogs()`. By default, `all_versions=False` returns only the latest available version of each catalogue.

In [5]:
catalogs = eso.list_catalogs(all_versions=False) # list the latest catalogue versions
catalogs[:10] # inspect the first few entries

['AMBRE_HARPS_V1',
 'AMBRE_UVES_V1',
 'AMBRE_V1',
 'AMUSED_MAIN_SOURCE_CAT_V1',
 'ATLASGAL_V1',
 'COSMOS2015_Laigle_v1_1b_latestV7_fits_V1',
 'COSMOS2020_CLASSIC_V2',
 'COSMOS2020_FARMER_V2',
 'EREBOS_RV_cat_fits_V1',
 'EREBOS_cat_fits_V1']

In [6]:
catalog_name in catalogs

True

# **Inspecting the catalogue schema**

As with the other query helpers, a good first step is to inspect the available columns with `help=True`. For catalogue cone searches, the important fields are the right ascension and declination columns.

The most reliable way to identify them is via their UCD values:

- `pos.eq.ra;meta.main` for right ascension
- `pos.eq.dec;meta.main` for declination

In [7]:
eso.query_catalog(catalog=catalog_name, help=True) # inspect the catalogue schema

INFO: 
Columns present in the table safcat.KiDS_DR4_1_ugriZYJHKs_cat_fits:
    column_name     datatype       unit                        ucd                 
------------------- -------- ---------------- -------------------------------------
                 ID     CHAR                                      meta.id;meta.main
          KIDS_TILE     CHAR                                                meta.id
         THELI_NAME     CHAR                                                meta.id
              SeqNr  INTEGER                                                meta.id
               SLID  INTEGER                                                meta.id
                SID  INTEGER                                                meta.id
          FLUX_AUTO     REAL            count                    phot.flux;em.opt.R
       FLUXERR_AUTO     REAL            count         stat.error;phot.flux;em.opt.R
           MAG_AUTO     REAL              mag                     phot.mag;em.opt.R
 

# **Finding the main coordinate columns**

Different catalogues may use different column names for the sky coordinates. 

The helper below queries `TAP_SCHEMA.columns` and looks for the main identifier, RA, and Dec columns from their UCD values.

In [8]:
MAIN_UCD_TO_KEY = {
    "meta.id;meta.main": "id",
    "pos.eq.ra;meta.main": "ra",
    "pos.eq.dec;meta.main": "dec",
}

def get_main_catalog_columns(catalog_name):
    """Return the main id/ra/dec columns for one catalogue table."""
    query = f"""
        SELECT column_name, ucd
        FROM TAP_SCHEMA.columns
        WHERE table_name = '{catalog_name}'
          AND ucd IN ('meta.id;meta.main', 'pos.eq.ra;meta.main', 'pos.eq.dec;meta.main')
    """

    rows = eso.query_tap(query, tap_endpoint="tap_cat")
    main_cols = {"id": None, "ra": None, "dec": None}

    for row in rows:
        key = MAIN_UCD_TO_KEY.get(row["ucd"])
        if key and main_cols[key] is None:
            main_cols[key] = row["column_name"]

    return main_cols

main_cols = get_main_catalog_columns(catalog_name)
main_cols

{'id': 'ID', 'ra': 'RAJ2000', 'dec': 'DECJ2000'}

# **Building the cone-search filter**

For now, the cone search must be expressed as an ADQL `CONTAINS(POINT, CIRCLE)` predicate and passed through `column_filters`.

Inside `query_catalog()`, the dictionary key becomes the left-hand side of the `WHERE` clause, so returning `{predicate: 1}` produces `CONTAINS(...) = 1`.

In [9]:
def make_catalog_cone_filter(ra, dec, radius, ra_column, dec_column):
    """Build a column_filters dictionary for a catalogue cone search."""
    ra_deg = ra.to_value(u.deg)
    dec_deg = dec.to_value(u.deg)
    radius_deg = radius.to_value(u.deg)

    predicate = (
        f"CONTAINS("
        f"POINT('', {ra_column}, {dec_column}), "
        f"CIRCLE('', {ra_deg}, {dec_deg}, {radius_deg})"
        f")"
    )

    return {predicate: 1}

column_filters = make_catalog_cone_filter(
    ra=coords.ra,
    dec=coords.dec,
    radius=radius,
    ra_column=main_cols["ra"],
    dec_column=main_cols["dec"],
)

# **Running the catalogue cone search**

Now we can submit the cone search with `query_catalog()`. 

To keep the output compact, we request a small subset of useful columns together with a row limit.

In [10]:
eso.ROW_LIMIT = 10

columns = [
    main_cols["id"],
    main_cols["ra"],
    main_cols["dec"],
    "KIDS_TILE",
    "MAG_AUTO",
    "MAGERR_AUTO",
]

table = eso.query_catalog(
    catalog=catalog_name,
    columns=columns,
    column_filters=column_filters,
)

table

ID,RAJ2000,DECJ2000,KIDS_TILE,MAG_AUTO,MAGERR_AUTO
,deg,deg,,mag,mag
object,float64,float64,object,float32,float32
KiDSDR4 J024629.784-301703.68,41.624103,-30.284356,KIDS_41.4_-30.2,21.47205,0.02750802
KiDSDR4 J024631.749-301701.50,41.632288,-30.283752,KIDS_41.4_-30.2,23.19277,0.07952795
KiDSDR4 J024632.188-301701.21,41.634118,-30.283671,KIDS_41.4_-30.2,22.41724,0.05056851
KiDSDR4 J024630.441-301700.97,41.62684,-30.283603,KIDS_41.4_-30.2,24.63857,0.1813087
KiDSDR4 J024632.254-301700.29,41.634394,-30.283415,KIDS_41.4_-30.2,22.93232,0.06367467
KiDSDR4 J024631.822-301659.43,41.632594,-30.283177,KIDS_41.4_-30.2,23.12575,0.07473277
KiDSDR4 J024631.129-301659.19,41.629708,-30.283109,KIDS_41.4_-30.2,24.51439,0.1454683
KiDSDR4 J024632.009-301658.40,41.633373,-30.282889,KIDS_41.4_-30.2,23.73499,0.08918835


At this stage it is often helpful to inspect the returned column descriptions, especially when switching to a different catalogue table.

In [11]:
print("Column Descriptions:\n" + "-" * 25)
for colname in table.colnames:
    desc = table[colname].info.description
    desc_str = desc if desc is not None else "[no description]"
    print(f"{colname:>15}: {desc_str}")

Column Descriptions:
-------------------------
             ID: ESO ID
        RAJ2000: Centroid sky position right ascension (J2000) (deg)
       DECJ2000: Centroid sky position declination (J2000) (deg)
      KIDS_TILE: Name of the pointing in AW convention
       MAG_AUTO: r-band magnitude (mag)
    MAGERR_AUTO: Error on MAG_AUTO (mag)


<hr style="border:2px solid #0281c9"> </hr>